# 02 - Data Analysis
This notebook focuses on using the processed fitness dataset to discover and explore trends in strength progression. This will be done by calculating an estimated one rep max for key exercises to have a standardized measure of strength, then using plots to visualize these trends in conjunction with related data such as workout frequency and bodyweight.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import LinearRegression

pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', 150)  # display up to 150 rows
pd.set_option('display.max_colwidth', None)  # increase column width
pd.set_option('display.float_format', '{:.2f}'.format)  # format floats to 2 decimal places 

In [ ]:
processed_df = pd.read_csv('../data/processed/processed_data.csv')

# convert columns to datetime again
processed_df['date'] = pd.to_datetime(processed_df['date'])
processed_df['start_time'] = pd.to_datetime(processed_df['start_time'])
processed_df['end_time'] = pd.to_datetime(processed_df['end_time'])

# ensure that the dataframe is sorted by the date
processed_df.sort_values(by='date', inplace=True)

processed_df.info()
processed_df.head()

### 2.1 - Determining Estimated One Rep Max
An estimated one rep max will be used as a standard measure of strength. This number will be obtained by using a lookup dictionary that gives a percentage of maximum strength based on reps and RPE for the set. These percentages are based on the findings of Mike Tuchscherer of Reactive Training Systems. This is then used in the `calculate_e1rm` function, after which a new column with the value is added to the processed data DataFrame.

In [ ]:
rpe_chart = {
    10: {1: 1.0, 2: 0.955, 3: 0.922, 4: 0.892, 5: 0.863, 6: 0.837, 7: 0.811, 8: 0.786, 9: 0.762, 10: 0.739, 11: 0.707, 12: 0.68, 13: 0.653, 14: 0.626, 15: 0.599},
    9.5: {1: 0.9775, 2: 0.9385, 3: 0.907, 4: 0.8775, 5: 0.85, 6: 0.824, 7: 0.7985, 8: 0.774, 9: 0.7505, 10: 0.723, 11: 0.6935, 12: 0.6665, 13: 0.6395, 14: 0.6125, 15: 0.5855},
    9: {1: 0.955, 2: 0.922, 3: 0.892, 4: 0.863, 5: 0.837, 6: 0.811, 7: 0.786, 8: 0.762, 9: 0.739, 10: 0.707, 11: 0.68, 12: 0.653, 13: 0.626, 14: 0.599, 15: 0.572},
    8.5: {1: 0.9385, 2: 0.907, 3: 0.8775, 4: 0.85, 5: 0.824, 6: 0.7985, 7: 0.774, 8: 0.7505, 9: 0.723, 10: 0.6935, 11: 0.6665, 12: 0.6395, 13: 0.6125, 14: 0.5855, 15: 0.5585},
    8: {1: 0.922, 2: 0.892, 3: 0.863, 4: 0.837, 5: 0.811, 6: 0.786, 7: 0.762, 8: 0.739, 9: 0.707, 10: 0.68, 11: 0.653, 12: 0.626, 13: 0.599, 14: 0.572, 15: 0.545},
    7.5: {1: 0.907, 2: 0.8775, 3: 0.85, 4: 0.824, 5: 0.7985, 6: 0.774, 7: 0.7505, 8: 0.723, 9: 0.6935, 10: 0.6665, 11: 0.6395, 12: 0.6125, 13: 0.5855, 14: 0.5585, 15: 0.5315},
    7: {1: 0.892, 2: 0.863, 3: 0.837, 4: 0.811, 5: 0.786, 6: 0.762, 7: 0.739, 8: 0.707, 9: 0.68, 10: 0.653, 11: 0.626, 12: 0.599, 13: 0.572, 14: 0.545, 15: 0.518},
    6: {1: 0.863, 2: 0.837, 3: 0.811, 4: 0.786, 5: 0.762, 6: 0.739, 7: 0.707, 8: 0.68, 9: 0.653, 10: 0.626, 11: 0.599, 12: 0.572, 13: 0.545, 14: 0.518, 15: 0.491}
}

In [ ]:
def calculate_e1rm(row):
    try:
        # obtain the information about the set necessary for calculating an estimated one rep max
        working_weight = row['working_weight_lbs']
        reps = row['reps']
        rpe = row['rpe']

        # disregard sets with over 15 reps since they are not accounted for in the chart and are irrelevant for determining strength
        if reps > 15:
            return None

        # return the estimated one rep max rounded to 1 decimal point
        percentage = rpe_chart[rpe][reps]
        return round(working_weight / percentage, 1)
    
    # return nothing when values are missing
    except (KeyError, TypeError):
        return None

In [ ]:
processed_df['estimated_1rm'] = processed_df.apply(calculate_e1rm, axis=1)
processed_df.info()  # for verifying the data was added properly
processed_df.head()

### 2.2 - Analyzing Bodyweight Trends
There are three underlying phases within the bodyweight data:
1. **Before Cancer Diagnosis:** During this period I was working out regularly, but without recording them. 
2. **During Cancer Treatment:** This period understandably caused a significant change in lifestyle where I engaged in very little activity.
3. **After Cancer Treatment:** Almost immediately after my final chemotherapy infusion I began working out again, and started tracking them as well. Therefore, all the workout data comes from this period.

Using vertical lines to mark these events, we can analyze how the weight changes differed within each phase.

In [ ]:
cancer_diagnosis = pd.to_datetime('02/15/2024')  # convert the date to a pd datetime object for analysis
first_recorded_workout = processed_df[processed_df['exercise_title'].notna()]['date'].min()  # find the first recorded workout date

In [ ]:
# add phase attribute to each row to show in visualization
def designate_phase(row):
    if row['date'] < cancer_diagnosis:
        return "Before Treatment"
    elif row ['date'] >= first_recorded_workout:
        return "After Treatment"
    else:
        return "During Treatment"

processed_df['phase'] = processed_df.apply(designate_phase, axis=1)

# bring the phase column to be after date
cols = processed_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('phase')))
processed_df = processed_df[cols]

In [ ]:
# fill in gaps in weight records with a straight line between existing values
processed_df['bodyweight_interpolated'] = processed_df['bodyweight_lbs'].interpolate()

# bring the bodyweight_interpolated column to be right after bodyweight_lbs
cols = processed_df.columns.tolist()
cols.insert(2, cols.pop(cols.index('bodyweight_interpolated')))
processed_df = processed_df[cols]

processed_df.head()

In [ ]:
# create a line chart that includes interpolation
bodyweight_chart = px.line(processed_df, 'date', 'bodyweight_interpolated', title="Bodyweight Over Time", labels={'date': 'Date', 'bodyweight_interpolated': 'Bodyweight (lbs)'}, color="phase")

# create a scatter plot that only includes recorded values
bodyweight_dots = px.scatter(processed_df, 'date', 'bodyweight_lbs')
bodyweight_dots.update_traces(
    marker=dict(size=5, opacity=0.5, color="gray")  # make dots more readable
)

# overlay the scatter plot on the line chart
bodyweight_chart.add_trace(bodyweight_dots.data[0])

# create vertical lines to split the visualization
bodyweight_chart.add_vline(cancer_diagnosis)
bodyweight_chart.add_vline(first_recorded_workout)

# adding labels for the vertical lines
bodyweight_chart.add_annotation(
    x=cancer_diagnosis,
    yref="paper",  # to have the label in the whitespace above chart
    y = 1.1,
    text="02/15/2024: Cancer Diagnosis",
    showarrow=False
)

bodyweight_chart.add_annotation(
    x=first_recorded_workout,
    yref="paper",
    y = 1.1,
    text="10/08/2024: First Recorded Workout",
    showarrow=False
)

bodyweight_chart.update_layout(
    xaxis_tickformat="%m/%Y"  # change date format from Jan 2000 to 01/2000
)

bodyweight_chart.show()

### 2.3 - Analyzing Strength Trends
There are several workouts that can be analyzed in depth, with varying levels of consistency in the data. Therefore, I will choose to analyze the Barbell Preacher Curl, an exercise that I have done consistently since I started recording my workouts. There is no data for this during cancer treatment, but there is still a visible upward trend that can be viewed in the data.

In [ ]:
preacher_curl_df = processed_df[processed_df['exercise_title'] == "Preacher Curl (Barbell)"]  # create dataframe of only preacher curls
preacher_curl_df = preacher_curl_df.dropna(subset=['estimated_1rm'])  # drop any rows without an e1rm (meaning no rpe or too many reps)
daily_max_e1rm_df = preacher_curl_df.groupby('date')['estimated_1rm'].max().reset_index()  # create dataframe with highest e1rm per workout

In [ ]:
# create a line chart to show change in e1rm
e1rm_chart = px.line(daily_max_e1rm_df, 'date', 'estimated_1rm', title="Estimated 1RM Over Time", labels={'date': 'Date', 'estimated_1rm': 'Estimated 1RM (lbs)'})

# create a scatter plot to show recorded values and get a trendline laid over line chart
e1rm_dots = px.scatter(daily_max_e1rm_df, 'date', 'estimated_1rm', trendline="ols")

    # customize scatter plot elements
e1rm_dots.update_traces(
    marker=dict(color="white", opacity=0.5),
    line=dict(color="rgba(255, 0, 0, 0.5)", dash="dashdot")
)

e1rm_chart.add_trace(e1rm_dots.data[0])
e1rm_chart.add_trace(e1rm_dots.data[1])
    
# date shown in mm/yyyy format
e1rm_chart.update_layout(
    xaxis_tickformat="%m/%Y"
)

e1rm_chart.show()

### 2.4 - Forecasting Future Strength Trends
Using Scikit-learn's linear regression model, we can predict how my strength progression will change over any indicated interval. I will choose a 90 day forecast, which should give me a good goal of an estimated 1-rep max to reach by that period. 

In [ ]:
forecast_df = daily_max_e1rm_df
            
# find first and last days exercise was recorded and create day counter
start_date = daily_max_e1rm_df['date'].min()
end_date = daily_max_e1rm_df['date'].max()
forecast_df['day_number'] = (forecast_df['date'] - start_date).dt.days

# variables for sklearn
X = forecast_df['day_number'].values.reshape(-1, 1)  # convert to 2d array
y = forecast_df['estimated_1rm'].values
            
# create range of dates
forecast_start_date = X[-1][0] + 1  # day after last recorded workout
forecast_end_date = forecast_start_date + 90  # create 90-day forecast
forecast_day_range = np.arange(forecast_start_date, forecast_end_date).reshape(-1, 1)  # numpy array of all days in chosen forecast period
                        
# create and train model then store forecasted values
model = LinearRegression()
model.fit(X, y)
forecasted_e1rm = model.predict(forecast_day_range)
forecasted_dates = pd.date_range(start=end_date + pd.Timedelta(days=1), periods=90, freq='D')  # starts on day after last recorded value and generates 90 dates that step day by day
            
# add forecasted data to existing chart and show chart
e1rm_chart.add_scatter(x=forecasted_dates, y=forecasted_e1rm, mode="lines", name = "Forecasted Data")

e1rm_chart.show()